# MV/LV coupling

This notebook shows the results of `pylovo-mv couple`, which checks the MV grids of a variant together with the
stored LV grids of the LV version:

1. **Design-point check** (one LV net at a time): a copy of the stored LV net with its own coincident loads and
   fixed station-busbar customer loads, the MV/LV tap of the station budget and the MV voltage of the load case with
   the HV/MV busbar at the lower edge of its control band gives the LV busbar voltage `coupled_lv_busbar_vm_pu` and
   the lowest LV voltage `coupled_lv_min_vm_pu`. The MV stage itself estimates these voltages analytically
   (`lv_busbar_vm_pu`, `lv_min_vm_pu`); the coupling verifies the estimate.
2. **Merged operating point**: every LV net fused into the MV net at its station, the LV loads scaled per MV feeder
   so that the feeder heads carry the loads of the MV design net, the station-busbar customers at the LV busbars.
   Result per station: `coupled_op_lv_busbar_vm_pu`; per district: convergence, slack power and voltage extremes.
   Every LV net is merged, also one that does not converge on its own (lines below 1 m get 1 m).

The design target of the LV busbar is the LV reference voltage of the LV version (`LV_REFERENCE_VOLTAGE_PU`, e.g.
0.96 p.u.) minus `lv_design_tolerance_pu`. A **band conflict** is a station where no MV/LV tap of the range meets
the target at the design point.

**Prerequisites**

- pylovo with the `notebooks` extra (`uv sync --extra notebooks`) and this notebook running on the kernel of the
  project environment (`.venv`), see `notebook_tutorials/README.md`.
- A `.env` in the repository root that points to the database with the MV results, and MV variants of the LV
  version (`pylovo-mv list --version <version>`; `0_mv_generation.ipynb` shows how to generate them).
- The notebook only reads from the database: its client runs read-only transactions.
- `pylovo-mv couple --version <version> --variant <variant>` for the variants to analyse; variants
  without coupling results are skipped.

These are stored results, not a new coupling run. After changing coupling logic, rerun `pylovo-mv couple` to update them. Results generated before the busbar-customer correction omit those customers from the design-point check.


In [ ]:
# Parameters
version_id = None  # LV version of the MV results; None: VERSION_ID of config/config_generation_lv.yaml
variants = None    # MV variants, e.g. ["<variant>"]; None: MV_VARIANT_ID if the version has it, else the
                   # newest done variant of the current MV stage version (only coupled ones are shown)
regions = None     # region parts to keep (1 = largest), e.g. [1]; None: all
plz = None         # keep only the districts with nodes in these PLZ; None: all
grid_id = None     # district of the detail map (id in the first coupled variant); None: the one with most stations
basemap = False    # OpenStreetMap tiles under the maps (needs internet access)
dbname = None  # expected database name of the .env; None: no check
repo_root = None   # repository root; None: two levels above this notebook

## Setup

The first cell changes to the repository root, so that pylovo reads the `.env` and `config/` of this checkout
(and writes its log to `log/`), and checks the database name when `dbname` is set. `mv_notebook_utils.py` next to
this notebook holds the shared helpers of the MV notebooks.

In [ ]:
import os
import sys
from pathlib import Path

if repo_root is None:
    here = Path.cwd().resolve()
    repo_root = next((p for p in (here, *here.parents)
                      if (p / "pyproject.toml").is_file() and (p / "src" / "pylovo").is_dir()), None)
    if repo_root is None:
        raise RuntimeError("Set repo_root to the MV checkout before running this notebook.")
repo_root = Path(repo_root).expanduser().resolve()

# A kernel registered for another checkout can silently import its pylovo installation.
import importlib.util

spec = importlib.util.find_spec("pylovo")
expected = repo_root / "src" / "pylovo" / "__init__.py"
if spec is None or spec.origin is None or Path(spec.origin).resolve() != expected:
    raise RuntimeError(
        f"Select the notebook kernel at {repo_root / '.venv' / 'bin' / 'python'}, "
        "restart it, then Run All. The current kernel imports pylovo from "
        f"{None if spec is None else spec.origin!r}."
    )
os.chdir(repo_root)
sys.path.insert(0, str(repo_root / "notebook_tutorials" / "mv_grids"))

from pylovo import config_loader

if dbname is not None:
    assert config_loader.DBNAME == dbname, f"the .env points to {config_loader.DBNAME!r}, not {dbname!r}"

import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

import mv_notebook_utils as mvu
from pylovo.plotting import mv_grid as mvplot
from pylovo.plotting.utils import get_color_map

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 400)
pd.set_option("display.max_colwidth", 200)

dbc = mvu.read_only_client()
version_id, variants, listing = mvu.resolve(dbc, version_id, variants)
print(f"Database {config_loader.DBNAME}, LV version {version_id}, MV variants: {', '.join(variants) or 'none'}")

In [ ]:
assert variants, "no MV variants to analyse"
plz_union = sorted({int(p) for p in listing.loc[listing["mv_variant_id"].isin(variants), "plz_list"].explode()})
parts = mvu.region_parts(dbc, plz_union)
data = {v: mvu.load_variant(dbc, version_id, v, parts=parts, regions=regions, plz=plz) for v in variants}
colors = get_color_map(variants)
districts = pd.concat([mvu.district_table(v) for v in data.values()], ignore_index=True)
rings = pd.concat([mvu.ring_table(v) for v in data.values()], ignore_index=True)
print(f"{len(parts)} region parts; districts per variant: "
      + ", ".join(f"{k} {len(v.grids)}" for k, v in data.items()))

from pylovo.mv_grid.parameters import LvVersionParameters

coupled = {n: v for n, v in data.items() if v.grids["coupled_status"].notna().any()}
for n in variants:
    if n not in coupled:
        print(f"{n}: no coupling results (pylovo-mv couple --version {version_id} --variant {n})")
print(f"Variants with coupling results: {', '.join(coupled) or 'none'}")
snapshot = dbc.lv_snapshot(version_id)
dbc.conn.rollback()
lv = LvVersionParameters.from_generation_parameters(version_id, snapshot)
limits = snapshot.get("power_flow_assessment") or {}
lv_min_pu, lv_max_pu = limits.get("min_vm_pu", 0.9), limits.get("max_vm_pu", 1.1)
target = lv.lv_reference_voltage_pu
stations = pd.concat([pd.DataFrame(v.nodes[v.nodes["kind"] == "station"].drop(columns=v.nodes.geometry.name))
                      .assign(variant=n) for n, v in coupled.items()], ignore_index=True) if coupled else None
print(f"LV reference voltage {target}, LV limits {lv_min_pu} to {lv_max_pu} p.u.")

## Coupling status

Districts by coupling status (`partial`: required checks failed or were skipped, a voltage budget is missing, or some LV nets were left out of the merged flow), then the counts of the
coupling per variant: stations, stations with an LV net, stations checked at the design point and merged into the
operating point, and the stations in each list of left-out LV nets.

In [ ]:
display(pd.crosstab(districts["variant"], districts["coupled_status"].fillna("not run")))
coupling = pd.concat([mvu.coupling_table(v) for v in coupled.values()], ignore_index=True) if coupled else None
if coupling is not None:
    count_columns = ["n_stations", "n_with_lv_grid", "n_checked", "n_merged"] + [f"n_{k}" for k in mvu.COUPLING_LISTS]
    totals = coupling.groupby("variant")[[c for c in count_columns if c in coupling]].sum()
    totals["runtime_s"] = coupling.groupby("variant")["runtime_s"].sum()
    display(mvu.pretty(totals.T, rows=True))

## LV busbar voltage at the design point

Analytic estimate of the MV stage against the coupled design point per station (`max_abs_diff_pu`, share within
0.005 p.u.), and the stations below the design target (with and without a band conflict).

In [ ]:
if coupled:
    rows = []
    for n, v in coupled.items():
        s = stations[stations["variant"] == n]
        tolerance = v.snapshot("taps.lv_design_tolerance_pu", 0.002)
        analytic, coupled_vm = s["lv_busbar_vm_pu"].astype(float), s["coupled_lv_busbar_vm_pu"].astype(float)
        diff = (coupled_vm - analytic).dropna()
        conflict = s["band_conflict"].astype("boolean").fillna(False).to_numpy()
        below = (coupled_vm < target - tolerance - 1e-9).to_numpy()
        rows.append({"variant": n, "n_stations": len(s), "n_coupled": int(coupled_vm.notna().sum()),
                     "design_target_pu": target - tolerance, "analytic_mean_pu": analytic.mean(),
                     "coupled_mean_pu": coupled_vm.mean(), "coupled_min_pu": coupled_vm.min(),
                     "coupled_max_pu": coupled_vm.max(), "max_abs_diff_pu": diff.abs().max(),
                     "agreement_share": (diff.abs() <= 0.005).mean(), "n_band_conflicts": int(conflict.sum()),
                     "n_below_target": int(below.sum()), "n_below_target_without_conflict": int((below & ~conflict).sum())})
    display(mvu.pretty(pd.DataFrame(rows).set_index("variant").T, rows=True))

In [ ]:
if coupled:
    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    values = stations["coupled_lv_busbar_vm_pu"].astype(float)
    edges = np.histogram_bin_edges(values.dropna(), bins=40)
    for n in coupled:
        sub = values[stations["variant"] == n].dropna()
        axes[0].hist(sub, bins=edges, histtype="step", linewidth=1.8, color=colors[n], label=f"{n} ({len(sub)})")
    tolerance = next(iter(coupled.values())).snapshot("taps.lv_design_tolerance_pu", 0.002)
    axes[0].axvline(target, color="#2c3e50", linestyle=":", label=f"LV reference {target:.3f} p.u.")
    axes[0].axvline(target - tolerance, color="#d62728", linestyle="--", label=f"design target {target - tolerance:.3f} p.u.")
    axes[0].set_xlabel("LV busbar voltage, coupled design point (p.u.)")
    axes[0].set_ylabel("Stations")
    axes[0].set_title("LV busbar voltage at the design point", fontweight="bold")
    axes[0].legend(fontsize=9)
    axes[0].grid(True, alpha=0.3)
    for n in coupled:
        s = stations[stations["variant"] == n]
        conflict = s["band_conflict"].astype("boolean").fillna(False).to_numpy()
        axes[1].scatter(s.loc[~conflict, "lv_busbar_vm_pu"], s.loc[~conflict, "coupled_lv_busbar_vm_pu"], s=12,
                        color=colors[n], alpha=0.6, label=f"{n}")
        axes[1].scatter(s.loc[conflict, "lv_busbar_vm_pu"], s.loc[conflict, "coupled_lv_busbar_vm_pu"], s=40,
                        marker="x", color=colors[n], label=f"{n}, band conflict")
    lo = np.nanmin(stations[["lv_busbar_vm_pu", "coupled_lv_busbar_vm_pu"]].astype(float).to_numpy())
    hi = np.nanmax(stations[["lv_busbar_vm_pu", "coupled_lv_busbar_vm_pu"]].astype(float).to_numpy())
    axes[1].plot([lo, hi], [lo, hi], color="#555555", linestyle=":", label="same value")
    axes[1].set_xlabel("LV busbar voltage, analytic budget (p.u.)")
    axes[1].set_ylabel("LV busbar voltage, coupled design point (p.u.)")
    axes[1].set_title("Analytic budget against coupled design point", fontweight="bold")
    axes[1].legend(fontsize=9)
    axes[1].grid(True, alpha=0.3)
    fig.tight_layout()

## Lowest LV voltage

The lowest voltage in the LV net at the design point against the lower LV limit of the LV version, from the
coupling and from the analytic budget of the MV stage. Stations whose LV net already violates the limits on its own
(LV design, flags `lv_voltage_violation` and `lv_not_converged`) are counted separately.

In [ ]:
if coupled:
    lv_flag = stations["flags"].map(lambda f: any(x in ("lv_voltage_violation", "lv_not_converged") for x in f)
                                    if isinstance(f, (list, tuple, np.ndarray)) else False)
    values = stations["coupled_lv_min_vm_pu"].astype(float)
    below = values < lv_min_pu
    analytic_below = stations["lv_min_vm_pu"].astype(float) < lv_min_pu
    fig, ax = plt.subplots(figsize=(10, 6))
    rows = []
    if values.dropna().empty:
        ax.text(0.5, 0.5, "No stored values", ha="center", va="center", transform=ax.transAxes)
    else:
        edges = np.histogram_bin_edges(values.dropna(), bins=40)
        for n in coupled:
            mask = (stations["variant"] == n).to_numpy()
            sub = values[mask].dropna()
            ax.hist(sub, bins=edges, histtype="step", linewidth=1.8, color=colors[n], label=f"{n} ({len(sub)})")
            rows.append({"variant": n, "n_stations": int(mask.sum()), "n_coupled_below": int(below[mask].sum()),
                         "n_coupled_below_without_lv_flag": int((below & ~lv_flag)[mask].sum()),
                         "n_analytic_below": int(analytic_below[mask].sum())})
    ax.axvline(lv_min_pu, color="#d62728", linestyle="--", label=f"limit {lv_min_pu:g} p.u.")
    ax.set_xlabel("Lowest LV voltage, design point (p.u.)")
    ax.set_ylabel("Stations")
    ax.set_title("Lowest LV voltage at the design point", fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    display(mvu.pretty(pd.DataFrame(rows, columns=["variant", "n_stations", "n_coupled_below",
                                                   "n_coupled_below_without_lv_flag", "n_analytic_below"])
                       .set_index("variant")))


## Band conflicts and MV/LV taps

The tap positions per variant (band conflicts hatched), the voltage budget chain of the stations and maps of the
coupled LV busbar voltage (all districts and one district, band conflicts crossed).

In [ ]:
if coupled:
    fig = mvplot.plot_tap_distribution({n: v.nodes for n, v in coupled.items()}, figsize=(12, 5))
    for n, v in coupled.items():
        fig = mvu.plot_voltage_budget(v, lv_reference_vm_pu=target or 0.96, lv_min_limit_pu=lv_min_pu,
                                      title=f"{n}: voltage budget of the stations")


In [ ]:
if coupled:
    first_name, first = next(iter(coupled.items()))
    detail = grid_id if grid_id is not None else mvu.default_district(first)
    for n, v in coupled.items():
        gid = detail if v is first else mvu.match_district(first, detail, v)[0]
        fig, axes = plt.subplots(1, 2, figsize=(22, 10))
        mvplot.plot_lv_busbar_voltage_map(v.layers, column="coupled_lv_busbar_vm_pu", ax=axes[0], basemap=basemap,
                                          title=f"{n}: LV busbar voltage at the coupled design point")
        mvplot.plot_lv_busbar_voltage_map(v.layers, column="coupled_lv_busbar_vm_pu", mv_grid_id=gid, ax=axes[1],
                                          basemap=basemap, title=f"{n}: district {gid}")
        fig.tight_layout()

## Merged operating point

Per district: convergence of the merged load flow, slack power at the HV/MV busbar, the lowest LV voltage of all LV
buses, the MV voltage difference to the MV-only net (a consistency check, should be a few mV), and the share of
stations whose LV busbar voltage at the operating point lies within 0.005 p.u. of the design point (the operating
point uses the coincident feeder loads instead of each station's own peak, so it differs).

In [ ]:
if coupling is not None:
    columns = {
        "coupled_status": "coupled_status", "n_merged": "n_merged", "n_not_merged": "n_not_merged",
        "operating_point.converged.load": "load_converged",
        "operating_point.slack_p_mw.load": "slack_p_load_mw", "operating_point.mv_only_slack_p_mw": "mv_only_slack_p_mw",
        "operating_point.min_lv_vm_pu": "op_min_lv_vm_pu",
        "operating_point.mv_vm_vs_mv_only.max_abs_diff_pu": "mv_vs_mv_only_max_abs_diff_pu",
        "operating_point.busbar_vs_design_point.share_within": "busbar_vs_design_point_share",
        "operating_point.busbar_vs_design_point.max_abs_diff_pu": "busbar_vs_design_point_max_abs_diff_pu",
        "operating_point.runtime_s": "op_runtime_s"}
    table = coupling.set_index(["variant", "mv_grid_id"]).reindex(columns=list(columns)).rename(columns=columns)
    display(mvu.pretty(table))
    fig, ax = plt.subplots(figsize=(10, 7))
    for n in coupled:
        s = stations[stations["variant"] == n]
        ax.scatter(s["coupled_lv_busbar_vm_pu"], s["coupled_op_lv_busbar_vm_pu"], s=12, color=colors[n], alpha=0.6,
                   label=n)
    lo = np.nanmin(stations[["coupled_lv_busbar_vm_pu", "coupled_op_lv_busbar_vm_pu"]].astype(float).to_numpy())
    hi = np.nanmax(stations[["coupled_lv_busbar_vm_pu", "coupled_op_lv_busbar_vm_pu"]].astype(float).to_numpy())
    ax.plot([lo, hi], [lo, hi], color="#555555", linestyle=":", label="same value")
    ax.set_xlabel("LV busbar voltage, coupled design point (p.u.)")
    ax.set_ylabel("LV busbar voltage, merged operating point (p.u.)")
    ax.set_title("Design point against merged operating point", fontweight="bold")
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

## LV grids with coupling issues

The stations in the lists of the coupling: LV nets that do not converge as stored (`lv_stored_not_converged`) or
at the design point (`design_not_converged`; they are merged all the same, variants coupled before 2026-10-19 left
them out as `not_merged`), stations without a budget or LV net, and LV errors. The LV grid is identified by
PLZ, `kcid` and `bcid`.

In [ ]:
left = pd.concat([mvu.coupling_left_out(v) for v in coupled.values()], ignore_index=True) if coupled else None
if left is None or left.empty:
    print("No LV grid was left out")
else:
    ids = sorted({int(g) for g in left["grid_result_id"].dropna()})
    dbc.cur.execute("SELECT grid_result_id, plz, kcid, bcid FROM pylovo.grid_result "
                    "WHERE version_id = %s AND grid_result_id = ANY(%s)", (version_id, ids))
    lv_grids = pd.DataFrame(dbc.cur.fetchall(), columns=["grid_result_id", "plz", "kcid", "bcid"])
    dbc.conn.rollback()
    left = left.merge(lv_grids, on="grid_result_id", how="left")
    display(left.pivot_table(index=["variant", "mv_grid_id", "mv_node_id", "grid_result_id", "plz", "kcid", "bcid",
                                    "sn_kva", "flags"], columns="list", values="item", aggfunc="size", fill_value=0))

## Station-busbar customers

MV customers below `customer_min_mv_kw` hang on the LV busbar of the nearest station (stage version 2 and later);
the coupling adds them as fixed loads at the LV busbar. Their number and load per variant and category, and the
LV busbar voltage of their stations against the other stations.

In [ ]:
if coupled:
    def record(stats):
        r = stats.get("busbar_customers") if isinstance(stats, dict) else None
        return r if isinstance(r, dict) else {}

    records = stations.assign(_r=stations["load_stats"].map(record))
    with_customers = records[records["_r"].map(lambda r: (r.get("n") or 0) > 0)]
    if with_customers.empty:
        print("No station-busbar customers in these variants (MV stage version 1 or no small MV customers)")
    else:
        summary = with_customers.groupby("variant").agg(
            n_stations=("mv_node_id", "size"), n_customers=("_r", lambda r: sum(x.get("n", 0) for x in r)),
            fixed_mw=("_r", lambda r: sum(x.get("kw", 0.0) for x in r) / 1000),
            installed_mw=("_r", lambda r: sum(x.get("installed_kw", 0.0) for x in r) / 1000),
            max_distance_m=("_r", lambda r: max(x.get("max_distance_m", 0.0) for x in r)))
        display(mvu.pretty(summary))
        cats = pd.DataFrame([{"variant": v, "category": c, "n": d.get("n"), "kw": d.get("kw")}
                             for v, r in zip(with_customers["variant"], with_customers["_r"])
                             for c, d in (r.get("categories") or {}).items()])
        display(cats.groupby(["variant", "category"]).sum())
        has = stations["mv_node_id"].isin(with_customers["mv_node_id"])
        display(stations.assign(busbar_customers=has).groupby(["variant", "busbar_customers"])[
            ["coupled_lv_busbar_vm_pu", "coupled_op_lv_busbar_vm_pu", "lv_busbar_vm_pu"]].mean().round(4))